# Analyse d'erreurs — profil `fr` V0.1 (protocole §7)

Annotation manuelle des 20 pires faux positifs et 20 pires faux négatifs du **dev split** (jamais du test). Taxonomie a priori §7.1. Exporte `docs/error_analysis_annotations_fr_v01.json` pour le rapport et le calcul du Cohen's kappa en cas de second annotateur.

## 1. Chargement des candidats

Source : `calibration/error_analysis_candidates_fr_v01.json` (extrait du dev split par calibrate.py).

In [ ]:
import json
from pathlib import Path

CANDIDATES = Path('calibration/error_analysis_candidates_fr_v01.json')
OUT = Path('docs/error_analysis_annotations_fr_v01.json')

data = json.loads(CANDIDATES.read_text(encoding='utf-8'))
fp, fn = data['false_positives'], data['false_negatives']
print(f'{len(fp)} FP candidats, {len(fn)} FN candidats')

## 2. Widget d'annotation

Pour chaque texte : coche **une** catégorie de la taxonomie, note libre optionnelle. Le bouton Enregistrer écrit le JSON d'annotations.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

FP_CATS = ['FP-1', 'FP-2', 'FP-3', 'FP-4', 'FP-5', 'FP-6', 'FP-autre']
FN_CATS = ['FN-1', 'FN-2', 'FN-3', 'FN-4', 'FN-5', 'FN-autre']

annotations = {}
try:
    annotations = json.loads(OUT.read_text(encoding='utf-8'))
except FileNotFoundError:
    pass

state = {'kind': None, 'idx': 0}

def show(kind, idx):
    state.update(kind=kind, idx=idx)
    items = fp if kind == 'fp' else fn
    cats = FP_CATS if kind == 'fp' else FN_CATS
    e = items[idx]
    display(Markdown(f"**{e['id']}** — {e['source']} — score {e['score']:.4f} — {e['length_chars']} chars "
                    f"({idx+1}/{len(items)})\n\n{e['text'][:1500]}"))
    prev = annotations.get(e['id'], {})
    cat = widgets.RadioButtons(options=cats, value=prev.get('category'), description='Catégorie')
    note = widgets.Text(value=prev.get('note', ''), description='Note')
    def save(_):
        annotations[e['id']] = {'category': cat.value, 'note': note.value}
        OUT.parent.mkdir(parents=True, exist_ok=True)
        OUT.write_text(json.dumps(annotations, ensure_ascii=False, indent=2), encoding='utf-8')
        print('enregistré', e['id'])
    btn = widgets.Button(description='Enregistrer')
    btn.on_click(save)
    nxt = widgets.Button(description='Suivant')
    nxt.on_click(lambda _: show(kind, (idx + 1) % len(items)))
    display(widgets.VBox([cat, note, widgets.HBox([btn, nxt])]))

print('Lancer show("fp", 0) pour annoter les faux positifs, show("fn", 0) pour les faux négatifs.')

## 3. Export — table de contingence Catégorie × Compte (§7.3)

Après annotation, ce bloc produit les tables FP et FN pour `docs/error_analysis_fr_v01.md`.

In [ ]:
from collections import Counter

done = {k: v for k, v in annotations.items() if v.get('category')}
fp_counts = Counter(v['category'] for k, v in done.items() if k.startswith('human'))
fn_counts = Counter(v['category'] for k, v in done.items() if k.startswith('ai'))
print('FP:', dict(fp_counts))
print('FN:', dict(fn_counts))
print(f'annotés: {len(done)}/40')